In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import mph
import mphsweepkit as msk

In [3]:
# Start the COMSOL client
client = mph.start()

In [4]:
# Load the model
model = client.load('cuboid_capacitor_solved.mph')

Load an already solved CascadedSweepModel

In [5]:
csm = msk.CascadedSweepModel(model, 'Cuboid Capacitor Study', show_param_names=True)

Initialized CascadedSweepModel
Study name: Cuboid Capacitor Study
Sweep Structure:
    - Geometry Sweep (BatchSweep) -> 'height', 'length', 'width'
      - Material Sweep (MaterialSweep) -> 'matsw.comp1.sw1'
        - Excitation Sweep (Parametric) -> 'voltage', 'core_temperature'
          - Coil Geometry Analysis (CoilCurrentCalculation)
            - Frequency Sweep (Frequency)
Loop names: ['Geometry Sweep', 'Material Sweep', 'Excitation Sweep', 'Coil Geometry Analysis', 'Frequency Sweep']
Loop lengths: [5, 3, 1, 1, 5]
Added 'geometry_idx' and 'internal_idx' columns to input_data. New shape: (75, 9)
--------------------------------
Data updated from MPh-model.
Input data shape: (75, 9)
Reset output data to shape of the input data: (75, 0)
Combined shape: (75, 9)


Post-processing of global equations

In [6]:
# Load the customized post-processing expressions from a JSON file
post_processing_exprs = msk.load_post_processing_exprs("global_post_processing_expressions.json", print_info=False)

# Perform post-processing of global data
csm.post_process_globals(post_processing_exprs)

# Show the first few rows of the combined data
# csm.combined_data.head()

# Save the input and output dataframes to CSV files
csm.save_global_data()

Saved result data to: X:\Till_data\repositories\MPhSweepKit\examples\studies\ferrite_plate_capacitor\global_data


Post-processing of fields on a selected domain

In [7]:
# All available selections can be printed with
csm.print_available_selections()

# Select a domain selection for post-processing.
# The selection name must exist in the COMSOL model:
selection_name = "cross-section"
selection_type = "bnd"
selection_domain_tag = csm._get_selection_tag(selection_name, selection_type=selection_type)
print(f"Selected domain tag:\n   {selection_domain_tag}")

# Choose a name for the dataset that will be created from the selection:
selection_dataset_name = "Solution Cross-Section"

# Create a dataset from the selection
csm.create_dataset_selection(selection_name, selection_type, selection_dataset_name)

# A dataset can be removed with
# csm.node_datasets.children()[-1].remove()

# Load customized post-processing expressions
post_processing_exprs = msk.load_post_processing_exprs("fields_post_processing_expressions.json", print_info=False)

# Compute and export the fields on all geometries in the selection dataset
# In the chosen sub-folder a new file will be created for each geometry in the selection dataset, containing the computed fields.
csm.export_fields_on_all_geometries(
    dataset_name=selection_dataset_name,
    post_processing_exprs=post_processing_exprs,
    sub_folder="Fields in Cross-Section"
)

Available selections:
   geom1_csel1_pnt cross-section
   geom1_csel1_edg cross-section
   geom1_csel1_bnd cross-section
   geom1_csel1_dom cross-section
Selected domain tag:
   geom1_csel1_bnd
Creating dataset 'Solution Cross-Section' from selection 'cross-section' of type 'bnd'.
Loop levels from COMSOL: [[1, 2, 3, 4, 5], [1, 2, 3], [1], [1], [1, 2, 3, 4, 5]]
Created sub-folder for field data: Fields in Cross-Section

Dataset to be exported:           'Solution Cross-Section'
    Check status of export nodes:
  Create export node:             'Magnetic Flux Density on Solution Cross-Section'
  Create export node:             'Electric Field on Solution Cross-Section'
    Exported field data to:       'Fields in Cross-Section/geometry_0_Magnetic Flux Density.txt'
    Exported field data to:       'Fields in Cross-Section/geometry_0_Electric Field.txt'

Dataset to be exported:           'Solution Cross-Section'
    Check status of export nodes:
        Overwrite export node:    'Magneti

Release Model

In [8]:
client.remove(model)
client.clear()
client.disconnect()